In [2]:
import numpy as np
import pandas as pd
import pickle
import time
import shap
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

# Données
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test_binary = pd.read_csv('../data/processed/y_test_binary.csv').squeeze()
y_train_binary = pd.read_csv('../data/processed/y_train_binary.csv').squeeze()
feature_names = X_train.columns.tolist()

# Modèles
with open('../models/random_forest_binary.pkl', 'rb') as f:
    rf_model = pickle.load(f)
with open('../models/decision_tree_binary.pkl', 'rb') as f:
    dt_model = pickle.load(f)
with open('../models/dnn_binary.pkl', 'rb') as f:
    dnn_model = pickle.load(f)

# Résultats XAI déjà calculés
with open('../models/shap_values_dict.pkl', 'rb') as f:
    shap_values_dict = pickle.load(f)
with open('../models/shap_local_dict.pkl', 'rb') as f:
    shap_local_dict = pickle.load(f)
with open('../models/lime_results.pkl', 'rb') as f:
    lime_results = pickle.load(f)
with open('../models/perm_results.pkl', 'rb') as f:
    perm_results = pickle.load(f)
with open('../models/dice_results.pkl', 'rb') as f:
    dice_results = pickle.load(f)
with open('../models/pdp_results.pkl', 'rb') as f:
    pdp_results = pickle.load(f)    

models = {'Random Forest': rf_model, 'Decision Tree': dt_model, 'DNN': dnn_model}

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [4]:
import matplotlib.pyplot as plt
import time
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import dice_ml

efficiency_results_200 = {}

X_efficiency = X_test.sample(n=200, random_state=42)
y_efficiency = y_test_binary.loc[X_efficiency.index]

top_features_idx = [feature_names.index(f) for f in ['spdx', 'spdy', 'posx', 'posy']]

for name, model in models.items():
    efficiency_results_200[name] = {}
    
    # SHAP Global
    start = time.time()
    if name != 'DNN':
        explainer = shap.TreeExplainer(model)
        explainer.shap_values(X_efficiency)
    else:
        bg = shap.sample(X_train, 50)
        explainer = shap.KernelExplainer(model.predict_proba, bg)
        explainer.shap_values(X_efficiency, nsamples=50)
    efficiency_results_200[name]['SHAP Global'] = round(time.time() - start, 1)
    
    # SHAP Local
    start = time.time()
    if name != 'DNN':
        explainer = shap.TreeExplainer(model)
        explainer.shap_values(X_efficiency.iloc[[0]])
    else:
        bg = shap.sample(X_train, 50)
        explainer = shap.KernelExplainer(model.predict_proba, bg)
        explainer.shap_values(X_efficiency.iloc[[0]], nsamples=50)
    efficiency_results_200[name]['SHAP Local'] = round(time.time() - start, 1)
    
    # LIME
    lime_exp = lime.lime_tabular.LimeTabularExplainer(
        X_train.values, feature_names=feature_names,
        class_names=['Normal', 'Attaque'], mode='classification'
    )
    start = time.time()
    for i in range(200):
        lime_exp.explain_instance(X_efficiency.values[i], model.predict_proba)
    efficiency_results_200[name]['LIME'] = round(time.time() - start, 1)
    
    # Permutation Importance
    start = time.time()
    permutation_importance(model, X_efficiency, y_efficiency, n_repeats=5, random_state=42)
    efficiency_results_200[name]['Permutation'] = round(time.time() - start, 1)
    
    # PDP
    start = time.time()
    PartialDependenceDisplay.from_estimator(
        model, X_efficiency, features=top_features_idx,
        feature_names=feature_names, n_jobs=-1
    )
    plt.close('all')
    efficiency_results_200[name]['PDP'] = round(time.time() - start, 1)
    
    # DICE
    start = time.time()
    d = dice_ml.Data(
        dataframe=pd.concat([X_efficiency, y_efficiency.rename('target')], axis=1),
        continuous_features=feature_names, outcome_name='target'
    )
    m = dice_ml.Model(model=model, backend='sklearn')
    exp = dice_ml.Dice(d, m, method='random')
    obs = X_efficiency.iloc[[0]]
    exp.generate_counterfactuals(obs, total_CFs=3, desired_class="opposite")
    efficiency_results_200[name]['DICE'] = round(time.time() - start, 1)
    
    print(f'{name} :')
    for method, t in efficiency_results_200[name].items():
        print(f'  {method} : {t}s')
    print()

100%|██████████| 1/1 [00:00<00:00,  1.96it/s]


Random Forest :
  SHAP Global : 2485.9s
  SHAP Local : 15.8s
  LIME : 16.3s
  Permutation : 3.4s
  PDP : 33.5s
  DICE : 0.6s



100%|██████████| 1/1 [00:00<00:00,  9.72it/s]


Decision Tree :
  SHAP Global : 13.1s
  SHAP Local : 0.1s
  LIME : 4.3s
  Permutation : 0.1s
  PDP : 1.4s
  DICE : 0.1s



100%|██████████| 1/1 [00:00<00:00,  9.70it/s]

DNN :
  SHAP Global : 4.6s
  SHAP Local : 0.1s
  LIME : 4.9s
  Permutation : 0.1s
  PDP : 1.1s
  DICE : 0.1s



In [7]:
from sklearn.metrics import accuracy_score

descriptive_accuracy_results = {}

for name, model in models.items():
    descriptive_accuracy_results[name] = {}
    
    # Performance de base sur le dataset complet
    base_acc = accuracy_score(y_test_binary, model.predict(X_test))
    descriptive_accuracy_results[name]['baseline'] = round(base_acc, 4)
    
    # --- SHAP Global ---
    shap_values = shap_values_dict[name]
    if isinstance(shap_values, list):
        imp_shap = np.abs(shap_values[1]).mean(axis=0)
    else:
        imp_shap = np.abs(shap_values).mean(axis=0)
    top3_shap = [feature_names[i] for i in np.argsort(imp_shap)[::-1][:3]]
    X_mod = X_test.copy()
    for col in top3_shap:
        X_mod[col] = 0
    acc = accuracy_score(y_test_binary, model.predict(X_mod))
    descriptive_accuracy_results[name]['SHAP Global'] = round(base_acc - acc, 4)
    
    # --- SHAP Local ---
    imp_shap_local = np.zeros(len(feature_names))
    for obs_name, obs_data in shap_local_dict[name].items():
        imp_shap_local += np.abs(obs_data['shap_values'])
    imp_shap_local /= len(shap_local_dict[name])
    top3_shap_local = [feature_names[i] for i in np.argsort(imp_shap_local)[::-1][:3]]
    X_mod = X_test.copy()
    for col in top3_shap_local:
        X_mod[col] = 0
    acc = accuracy_score(y_test_binary, model.predict(X_mod))
    descriptive_accuracy_results[name]['SHAP Local'] = round(base_acc - acc, 4)
    
    # --- LIME ---
    imp_lime = np.zeros(len(feature_names))
    for obs_name, obs_data in lime_results[name].items():
        for feat, val in obs_data['features']:
            for j, fname in enumerate(feature_names):
                if fname in feat:
                    imp_lime[j] += abs(val)
    top3_lime = [feature_names[i] for i in np.argsort(imp_lime)[::-1][:3]]
    X_mod = X_test.copy()
    for col in top3_lime:
        X_mod[col] = 0
    acc = accuracy_score(y_test_binary, model.predict(X_mod))
    descriptive_accuracy_results[name]['LIME'] = round(base_acc - acc, 4)
    
    # --- Permutation Importance ---
    imp_perm = np.array(perm_results[name]['importances_mean'])
    top3_perm = [feature_names[i] for i in np.argsort(imp_perm)[::-1][:3]]
    X_mod = X_test.copy()
    for col in top3_perm:
        X_mod[col] = 0
    acc = accuracy_score(y_test_binary, model.predict(X_mod))
    descriptive_accuracy_results[name]['Permutation'] = round(base_acc - acc, 4)
    
    # --- PDP ---
    pdp_data = pdp_results[name]
    pdp_importance = []
    for i, (avg, grid) in enumerate(pdp_data['pd_results']):
        pdp_importance.append(np.max(avg) - np.min(avg))
    top3_pdp = [pdp_data['features'][i] for i in np.argsort(pdp_importance)[::-1][:3]]
    X_mod = X_test.copy()
    for col in top3_pdp:
        X_mod[col] = 0
    acc = accuracy_score(y_test_binary, model.predict(X_mod))
    descriptive_accuracy_results[name]['PDP'] = round(base_acc - acc, 4)
    
    print(f'{name} — Baseline : {base_acc:.4f}')
    for method, baisse in descriptive_accuracy_results[name].items():
        if method != 'baseline':
            print(f'  Baisse {method} : {baisse}')
    print()

Random Forest — Baseline : 0.9007
  Baisse SHAP Global : 0.4707
  Baisse SHAP Local : 0.468
  Baisse LIME : 0.4707
  Baisse Permutation : 0.4736
  Baisse PDP : 0.419

Decision Tree — Baseline : 0.8649
  Baisse SHAP Global : 0.3947
  Baisse SHAP Local : 0.3838
  Baisse LIME : 0.412
  Baisse Permutation : 0.3729
  Baisse PDP : 0.2718

DNN — Baseline : 0.8272
  Baisse SHAP Global : 0.4017
  Baisse SHAP Local : 0.3938
  Baisse LIME : 0.3908
  Baisse Permutation : 0.4114
  Baisse PDP : 0.3908



In [10]:
sparsity_results = {}

for name, model in models.items():
    sparsity_results[name] = {}
    
    # --- SHAP Global ---
    shap_values = shap_values_dict[name]
    if isinstance(shap_values, list):
        imp = np.abs(shap_values[1]).mean(axis=0)
    else:
        imp = np.abs(shap_values).mean(axis=0)
    cumulative = np.cumsum(np.sort(imp)[::-1]) / np.sum(imp)
    sparsity_results[name]['SHAP Global'] = int(np.argmax(cumulative >= 0.80) + 1)
    
    # --- SHAP Local ---
    imp_local = np.zeros(len(feature_names))
    for obs_data in shap_local_dict[name].values():
        imp_local += np.abs(obs_data['shap_values'])
    imp_local /= len(shap_local_dict[name])
    cumulative = np.cumsum(np.sort(imp_local)[::-1]) / np.sum(imp_local)
    sparsity_results[name]['SHAP Local'] = int(np.argmax(cumulative >= 0.80) + 1)
    
    # --- LIME ---
    imp_lime = np.zeros(len(feature_names))
    for obs_data in lime_results[name].values():
        for feat, val in obs_data['features']:
            for j, fname in enumerate(feature_names):
                if fname in feat:
                    imp_lime[j] += abs(val)
    imp_lime /= len(lime_results[name])
    cumulative = np.cumsum(np.sort(imp_lime)[::-1]) / np.sum(imp_lime)
    sparsity_results[name]['LIME'] = int(np.argmax(cumulative >= 0.80) + 1)
    
    # --- DICE ---
    dice_sparsity = []
    for obs_name, cf in dice_results[name].items():
        try:
            cf_df = cf.cf_examples_list[0].final_cfs_df
            original = cf.cf_examples_list[0].test_instance_df
            for _, row in cf_df.iterrows():
                n_changed = sum(
                abs(row[feature_names] - original[feature_names].values[0]) > 0.01
            )
            dice_sparsity.append(n_changed)
        except:
            pass
    sparsity_results[name]['DICE'] = round(np.mean(dice_sparsity), 2) if dice_sparsity else None
    
    print(f'{name} :')
    print(f'  SHAP Global : {sparsity_results[name]["SHAP Global"]} features pour 80%')
    print(f'  SHAP Local  : {sparsity_results[name]["SHAP Local"]} features pour 80%')
    print(f'  LIME        : {sparsity_results[name]["LIME"]} features pour 80%')
    print(f'  DICE        : {sparsity_results[name]["DICE"]} features modifiées en moyenne')

Random Forest :
  SHAP Global : 9 features pour 80%
  SHAP Local  : 10 features pour 80%
  LIME        : 8 features pour 80%
  DICE        : 1.75 features modifiées en moyenne
Decision Tree :
  SHAP Global : 8 features pour 80%
  SHAP Local  : 10 features pour 80%
  LIME        : 8 features pour 80%
  DICE        : 1.25 features modifiées en moyenne
DNN :
  SHAP Global : 9 features pour 80%
  SHAP Local  : 10 features pour 80%
  LIME        : 10 features pour 80%
  DICE        : 1.5 features modifiées en moyenne


In [4]:
from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

stability_results = {}

def jaccard(set_a, set_b):
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 1.0

for name, model in models.items():
    stability_results[name] = {}
    
    # --- SHAP Global ---
    top3_runs = []
    for run in range(5):
        X_run = X_test.sample(n=50, random_state=run)
        if name != 'DNN':
            explainer = shap.TreeExplainer(model)
            sv = explainer.shap_values(X_run)
            imp = np.abs(sv[1] if isinstance(sv, list) else sv).mean(axis=0)
        else:
            bg = shap.sample(X_train, 50)
            explainer = shap.KernelExplainer(model.predict_proba, bg)
            sv = explainer.shap_values(X_run.iloc[:10], nsamples=50)
            imp = np.abs(sv[1]).mean(axis=0)
        top3_runs.append(set(np.argsort(imp)[::-1][:3]))
    scores = [jaccard(top3_runs[i], top3_runs[j]) for i in range(5) for j in range(i+1, 5)]
    stability_results[name]['SHAP Global'] = round(np.mean(scores), 4)
    
    # --- SHAP Local ---
    top3_local_runs = []
    for obs_data in shap_local_dict[name].values():
        imp = np.abs(obs_data['shap_values'])
        top3_local_runs.append(set(np.argsort(imp)[::-1][:3]))
    scores = [jaccard(top3_local_runs[i], top3_local_runs[j]) 
              for i in range(len(top3_local_runs)) 
              for j in range(i+1, len(top3_local_runs))]
    stability_results[name]['SHAP Local'] = round(np.mean(scores), 4)
    
    # --- Permutation Importance ---
    top3_perm_runs = []
    for run in range(5):
        X_run = X_test.sample(n=100, random_state=run)
        y_run = y_test_binary.loc[X_run.index]
        result = permutation_importance(model, X_run, y_run, n_repeats=3, random_state=run)
        top3_perm_runs.append(set(np.argsort(result.importances_mean)[::-1][:3]))
    scores = [jaccard(top3_perm_runs[i], top3_perm_runs[j]) for i in range(5) for j in range(i+1, 5)]
    stability_results[name]['Permutation'] = round(np.mean(scores), 4)
    
    # --- PDP ---
    top3_pdp_runs = []
    for run in range(5):
        X_run = X_test.sample(n=100, random_state=run)
        display = PartialDependenceDisplay.from_estimator(
            model, X_run, features=list(range(len(feature_names))),
            feature_names=feature_names, n_jobs=-1
        )
        plt.close('all')
        pdp_imp = [np.max(pd.average) - np.min(pd.average) for pd in display.pd_results]
        top3_pdp_runs.append(set(np.argsort(pdp_imp)[::-1][:3]))
    scores = [jaccard(top3_pdp_runs[i], top3_pdp_runs[j]) for i in range(5) for j in range(i+1, 5)]
    stability_results[name]['PDP'] = round(np.mean(scores), 4)
    
    # --- DICE ---
    dice_runs = []
    for obs_name, cf in dice_results[name].items():
        try:
            cf_df = cf.cf_examples_list[0].final_cfs_df
            original = cf.cf_examples_list[0].test_instance_df
            changed = set(feat for feat in feature_names
                         if abs(float(cf_df[feat].iloc[0]) - float(original[feat].values[0])) > 0.01)
            if changed:
                dice_runs.append(changed)
        except:
            pass
    if len(dice_runs) >= 2:
        scores = [jaccard(dice_runs[i], dice_runs[j]) 
                  for i in range(len(dice_runs)) 
                  for j in range(i+1, len(dice_runs))]
        stability_results[name]['DICE'] = round(np.mean(scores), 4)
    else:
        stability_results[name]['DICE'] = None

    print(f'{name} :')
    for method, stab in stability_results[name].items():
        print(f'  {method} : {stab} (1.0 = parfaitement stable)')
    print()

Random Forest :
  SHAP Global : 0.8 (1.0 = parfaitement stable)
  SHAP Local : 0.15 (1.0 = parfaitement stable)
  Permutation : 1.0 (1.0 = parfaitement stable)
  PDP : 0.46 (1.0 = parfaitement stable)
  DICE : 0.0556 (1.0 = parfaitement stable)

Decision Tree :
  SHAP Global : 0.8 (1.0 = parfaitement stable)
  SHAP Local : 0.25 (1.0 = parfaitement stable)
  Permutation : 1.0 (1.0 = parfaitement stable)
  PDP : 0.52 (1.0 = parfaitement stable)
  DICE : 0.0833 (1.0 = parfaitement stable)



100%|██████████| 10/10 [00:00<00:00, 46.62it/s]


DNN :
  SHAP Global : 0.16 (1.0 = parfaitement stable)
  SHAP Local : 0.15 (1.0 = parfaitement stable)
  Permutation : 0.56 (1.0 = parfaitement stable)
  PDP : 0.7 (1.0 = parfaitement stable)
  DICE : 0.0556 (1.0 = parfaitement stable)



In [11]:
completeness_results = {}

for name, model in models.items():
    completeness_results[name] = {}
    
    # --- SHAP Global ---
    shap_values = shap_values_dict[name]
    if isinstance(shap_values, list):
        imp = np.abs(shap_values[1]).mean(axis=0)
    else:
        imp = np.abs(shap_values).mean(axis=0)
    sorted_imp = np.sort(imp)[::-1]
    completeness_results[name]['SHAP Global'] = round(
        np.sum(sorted_imp[:5]) / np.sum(sorted_imp), 4)
    
    # --- SHAP Local ---
    imp_local = np.zeros(len(feature_names))
    for obs_data in shap_local_dict[name].values():
        imp_local += np.abs(obs_data['shap_values'])
    imp_local /= len(shap_local_dict[name])
    sorted_imp = np.sort(imp_local)[::-1]
    completeness_results[name]['SHAP Local'] = round(
        np.sum(sorted_imp[:5]) / np.sum(sorted_imp), 4)
    # --- LIME ---
    imp_lime = np.zeros(len(feature_names))
    for obs_data in lime_results[name].values():
        for feat, val in obs_data['features']:
            for j, fname in enumerate(feature_names):
                if fname in feat:
                    imp_lime[j] += abs(val)
    imp_lime /= len(lime_results[name])
    sorted_imp = np.sort(imp_lime)[::-1]
    completeness_results[name]['LIME'] = round(
        np.sum(sorted_imp[:5]) / np.sum(sorted_imp), 4)
    
    print(f'{name} :')
    for method, comp in completeness_results[name].items():
        print(f'  {method} : {comp} (1.0 = top 5 features couvrent tout)')
    print()

Random Forest :
  SHAP Global : 0.5803 (1.0 = top 5 features couvrent tout)
  SHAP Local : 0.5111 (1.0 = top 5 features couvrent tout)
  LIME : 0.6726 (1.0 = top 5 features couvrent tout)

Decision Tree :
  SHAP Global : 0.6264 (1.0 = top 5 features couvrent tout)
  SHAP Local : 0.5688 (1.0 = top 5 features couvrent tout)
  LIME : 0.6575 (1.0 = top 5 features couvrent tout)

DNN :
  SHAP Global : 0.5282 (1.0 = top 5 features couvrent tout)
  SHAP Local : 0.5484 (1.0 = top 5 features couvrent tout)
  LIME : 0.554 (1.0 = top 5 features couvrent tout)



In [12]:
consistency_results = {}

for name in models.keys():
    consistency_results[name] = {}
    
    # Top 3 features selon chaque méthode
    
    # SHAP Global
    shap_values = shap_values_dict[name]
    if isinstance(shap_values, list):
        imp_shap = np.abs(shap_values[1]).mean(axis=0)
    else:
        imp_shap = np.abs(shap_values).mean(axis=0)
    top3_shap = set(np.argsort(imp_shap)[::-1][:3])
    
    # Permutation Importance
    imp_perm = np.array(perm_results[name]['importances_mean'])
    top3_perm = set(np.argsort(imp_perm)[::-1][:3])
    
    # PDP
    pdp_data = pdp_results[name]
    pdp_imp = [np.max(avg) - np.min(avg) for avg, grid in pdp_data['pd_results']]
    top3_pdp = set(np.argsort(pdp_imp)[::-1][:3])
    
    # SHAP Local
    imp_local = np.zeros(len(feature_names))
    for obs_data in shap_local_dict[name].values():
        imp_local += np.abs(obs_data['shap_values'])
    imp_local /= len(shap_local_dict[name])
    top3_shap_local = set(np.argsort(imp_local)[::-1][:3])
    
    # LIME
    imp_lime = np.zeros(len(feature_names))
    for obs_data in lime_results[name].values():
        for feat, val in obs_data['features']:
            for j, fname in enumerate(feature_names):
                if fname in feat:
                    imp_lime[j] += abs(val)
    top3_lime = set(np.argsort(imp_lime)[::-1][:3])
    
    # --- Consistency Globale (SHAP Global, Permutation, PDP) ---
    global_methods = {
        'SHAP Global': top3_shap,
        'Permutation': top3_perm,
        'PDP': top3_pdp
    }
    global_intersections = []
    print(f'{name} — Méthodes Globales :')
    gm = list(global_methods.keys())
    for i in range(len(gm)):
        for j in range(i+1, len(gm)):
            inter = len(global_methods[gm[i]] & global_methods[gm[j]])
            global_intersections.append(inter)
            print(f'  {gm[i]} ∩ {gm[j]} : {inter}/3')
    consistency_global = round(np.mean(global_intersections) / 3, 4)
    consistency_results[name]['global'] = consistency_global
    print(f'  → Consistency Globale : {consistency_global}')
    
    # --- Consistency Locale (SHAP Local, LIME) ---
    local_inter = len(top3_shap_local & top3_lime)
    consistency_local = round(local_inter / 3, 4)
    consistency_results[name]['local'] = consistency_local
    print(f'\n{name} — Méthodes Locales :')
    print(f'  SHAP Local ∩ LIME : {local_inter}/3')
    print(f'  → Consistency Locale : {consistency_local}')
    print()

Random Forest — Méthodes Globales :
  SHAP Global ∩ Permutation : 2/3
  SHAP Global ∩ PDP : 0/3
  Permutation ∩ PDP : 1/3
  → Consistency Globale : 0.3333

Random Forest — Méthodes Locales :
  SHAP Local ∩ LIME : 2/3
  → Consistency Locale : 0.6667

Decision Tree — Méthodes Globales :
  SHAP Global ∩ Permutation : 2/3
  SHAP Global ∩ PDP : 0/3
  Permutation ∩ PDP : 1/3
  → Consistency Globale : 0.3333

Decision Tree — Méthodes Locales :
  SHAP Local ∩ LIME : 1/3
  → Consistency Locale : 0.3333

DNN — Méthodes Globales :
  SHAP Global ∩ Permutation : 2/3
  SHAP Global ∩ PDP : 0/3
  Permutation ∩ PDP : 0/3
  → Consistency Globale : 0.2222

DNN — Méthodes Locales :
  SHAP Local ∩ LIME : 1/3
  → Consistency Locale : 0.3333



In [14]:
dice_metrics_results = {}

for name, model in models.items():
    dice_metrics_results[name] = {}
    
    validity_scores = []
    diversity_scores = []
    
    for obs_name, cf in dice_results[name].items():
        try:
            cf_df = cf.cf_examples_list[0].final_cfs_df
            original = cf.cf_examples_list[0].test_instance_df
            original_pred = model.predict(original[feature_names])[0]
            
            # --- Validity ---
            # Est-ce que chaque contrefactuel change vraiment la prédiction ?
            valid_count = 0
            for _, row in cf_df.iterrows():
                cf_pred = model.predict([row[feature_names].values])[0]
                if cf_pred != original_pred:
                    valid_count += 1
            validity = valid_count / len(cf_df)
            validity_scores.append(validity)
            
            # --- Diversity ---
            # Est-ce que les contrefactuels sont différents entre eux ?
            cf_values = cf_df[feature_names].values
            pairwise_distances = []
            for i in range(len(cf_values)):
                for j in range(i+1, len(cf_values)):
                    dist = np.mean(np.abs(cf_values[i] - cf_values[j]))
                    pairwise_distances.append(dist)
            diversity = np.mean(pairwise_distances) if pairwise_distances else 0
            diversity_scores.append(diversity)
            
        except Exception as e:
            print(f'  Erreur {obs_name} : {e}')
    
    dice_metrics_results[name]['Validity'] = round(np.mean(validity_scores), 4)
    dice_metrics_results[name]['Diversity'] = round(np.mean(diversity_scores), 4)
    
    print(f'{name} :')
    print(f'  Validity  : {dice_metrics_results[name]["Validity"]} (1.0 = tous les contrefactuels changent la prédiction)')
    print(f'  Diversity : {dice_metrics_results[name]["Diversity"]} (plus élevé = contrefactuels plus diversifiés)')
    print()

Random Forest :
  Validity  : 1.0 (1.0 = tous les contrefactuels changent la prédiction)
  Diversity : 0.9038 (plus élevé = contrefactuels plus diversifiés)

Decision Tree :
  Validity  : 0.9167 (1.0 = tous les contrefactuels changent la prédiction)
  Diversity : 0.9772 (plus élevé = contrefactuels plus diversifiés)

DNN :
  Validity  : 1.0 (1.0 = tous les contrefactuels changent la prédiction)
  Diversity : 0.9417 (plus élevé = contrefactuels plus diversifiés)



In [17]:
all_metrics = {
    'efficiency': efficiency_results,
    'descriptive_accuracy': descriptive_accuracy_results,
    'sparsity': sparsity_results,
    'stability': stability_results,
    'completeness': completeness_results,
    'consistency': consistency_results,
    'dice_metrics': dice_metrics_results
}

with open('../models/metrics_xai.pkl', 'wb') as f:
    pickle.dump(all_metrics, f)